# Preprocessing walkthrough

Owner: Ana Valderrama — step-by-step `data_prep` functions.

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
ROOT = _cwd if (_cwd / "config.yaml").exists() else _cwd.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data_prep import (
    project_root,
    aggregate_channels,
    apply_adstock,
    fill_date_gaps,
    fill_vertical_nulls,
    handle_spend_nulls,
    load_config,
    load_raw,
    normalize_currency,
    remove_duplicates,
    split_train_test,
    validate_spend_non_negative,
    winsorize_spend,
)

ROOT = project_root()
config = load_config()
raw = ROOT / "data" / "raw" / "uploaded_dataset.csv"
if not raw.exists():
    raise FileNotFoundError("Create data/raw/uploaded_dataset.csv via upload or 01_eda.")

df = load_raw(str(raw), config)
df = remove_duplicates(df, config)
df = normalize_currency(df, config["fx_rates"])
df = fill_vertical_nulls(df, config)
df = handle_spend_nulls(df, config=config)
df = validate_spend_non_negative(df)
df = winsorize_spend(df, config=config)
df = fill_date_gaps(df, config)
df = apply_adstock(df, config["adstock"]["decay_rates"], config)
df = aggregate_channels(df, config)
train, test = split_train_test(df, config=config)
print("Ready:", df.shape, "Train:", train.shape, "Test:", test.shape)
df.head()